# 🚀 PostgreSQL (pgvector) 기반 공통 RAG 검색 파이프라인 가이드 (기초)

본 노트북은 **BIST Mini-2 프로젝트**에서 사용되는 공통 RAG(Retrieval-Augmented Generation) 검색 엔진(`CommonRagPipeline`)의 기본 원리와 동작 방식을 이해하기 위한 기초 학습 가이드입니다.

이 노트북을 통해 데이터베이스에서 질의어(Query)를 기반으로 관련 문헌 조각(Chunk)을 유사도 순으로 검색해내는 전체 메커니즘을 파헤쳐 봅니다.

---

## 📌 RAG 검색 핵심 흐름
1. **[설정 로드]** `backend/.env` 파일로부터 DB 접속정보 및 OpenAI API Key 로드
2. **[임베딩 모델 초기화]** OpenAI `text-embedding-3-large` (3072차원) 모델 준비
3. **[벡터 데이터베이스 연결]** LangChain `PGVector` 클래스를 이용해 지정 도메인 컬렉션 연결
4. **[유사도 검색 실행]** 코사인 거리(Distance) 기반 유사도 점수 산출 및 결과 정렬

## 1단계. 백엔드 환경 설정 로드

백엔드 설정 모듈(`api.common.config`)과 동일하게 동작할 수 있도록 `backend/.env` 파일을 로드하고 필요한 환경 변수를 검증합니다.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1) 프로젝트 루트 경로 및 백엔드 경로 확인
# 현재 폴더 위치: notebooks/rag_pipeline/
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

# 백엔드 모듈 임포트를 위해 sys.path에 추가
if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 2) backend/.env 환경 변수 파일 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정을 성공적으로 로드했습니다. ({env_path})")
else:
    print("ℹ️ backend/.env 파일을 찾지 못했습니다. 시스템 환경 변수를 사용합니다.")

# 3) 중요 설정 검증
openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

print(f"🔑 OpenAI API Key 설정 여부: {'설정됨(Yes)' if openai_key else '설정안됨(No)'}")
print(f"🗄️ Database URL: {database_url[:35]}...")

✅ 백엔드 환경 설정을 성공적으로 로드했습니다. (/Users/pileuszu/Repos/bist-mini-2/backend/.env)
🔑 OpenAI API Key 설정 여부: 설정됨(Yes)
🗄️ Database URL: postgresql+asyncpg://postgres:postg...


## 2단계. 임베딩 모델 및 PGVector 연결 설정

RAG 서비스 모듈(`api.common.rag_pipeline`)의 임베딩 규격과 데이터베이스 연결 정보를 정의하고 인스턴스를 초기화합니다.

In [2]:
from langchain_openai import OpenAIEmbeddings

# text-embedding-3-large 모델 고정 규격 사용
EMBED_MODEL = "text-embedding-3-large"
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=openai_key
)

# 데이터베이스 연결 정보 설정
# 백엔드 Pydantic Settings 기준과 동일하게 asyncpg -> psycopg_async로 변경
pgvector_url = database_url.replace("postgresql+asyncpg://", "postgresql+psycopg_async://")

print(f"✅ 임베딩 엔진 준비 완료 (모델명: {EMBED_MODEL})")
print(f"✅ PGVector 연결 주소 구성 완료: {pgvector_url[:35]}...")

✅ 임베딩 엔진 준비 완료 (모델명: text-embedding-3-large)
✅ PGVector 연결 주소 구성 완료: postgresql+psycopg_async://postgres...


## 3단계. RAG 유사도 검색 함수 정의 (Similarity Search)

백엔드 공통 RAG 파이프라인에서 핵심적으로 수행하는 `asimilarity_search_with_score` 및 **코사인 유사도(Cosine Similarity) 점수 계산** 로직을 함수로 구현합니다.

In [3]:
from langchain_postgres import PGVector

DOMAIN_COLLECTIONS = {
    "bio": "bio_embeddings",
    "cs": "cs_embeddings",
    "astronomy": "astronomy_embeddings"
}

async def run_similarity_search(domain: str, query: str, k: int = 3):
    """지정된 도메인의 pgvector 테이블에서 유사한 문서를 검색합니다."""
    if domain not in DOMAIN_COLLECTIONS:
        raise ValueError(f"지원하지 않는 도메인입니다: {domain}")
        
    collection_name = DOMAIN_COLLECTIONS[domain]
    print(f"🔎 RAG 검색 개시 | 도메인: {domain} ({collection_name}) | 검색어: '{query}'")
    
    # 1. PGVector 클라이언트 연결 선언
    vectorstore = PGVector(
        embeddings=embeddings_model,
        collection_name=collection_name,
        connection=pgvector_url,
        async_mode=True,
    )
    
    # 2. 비동기 유사도 검색 실행 (거리 점수 포함 반환)
    # 결과 형식: List[Tuple[Document, float]] (여기서 float은 L2 또는 코사인 거리 값)
    results = await vectorstore.asimilarity_search_with_score(query, k=k)
    
    formatted_results = []
    for doc, score in results:
        meta = doc.metadata or {}
        arxiv_id = meta.get("arxiv_id") or meta.get("doc_id") or ""
        title = meta.get("title", "")
        
        # 3. 코사인 유사도 점수 환산
        # pgvector의 distance score는 (1.0 - 유사도) 이므로, 유사도는 (1.0 - score)로 역산합니다.
        similarity = round(1.0 - score, 4)
        
        formatted_results.append({
            "doc_id": arxiv_id,
            "title": title,
            "text_chunk": doc.page_content,
            "score": similarity
        })
        
    return formatted_results

## 4단계. 실시간 검색 테스트 및 결과 시각화

구현된 RAG 검색 로직을 동작시키고, 최종 검출되는 논문 정보와 매칭 점수를 가시화합니다.
* 실제 DB 연결이 차단되어 있거나 오류가 발생할 경우, 동작 흐름을 시각적으로 확인할 수 있는 Mock 데이터 대체(Fallback) 처리도 함께 설계되어 있습니다.

In [4]:
# 컴퓨터과학(cs) 도메인에 유사도 검색 수행 시뮬레이션
test_query = "Neural network models and sequence alignment in computational biology"
search_results = await run_similarity_search("cs", test_query, k=2)

# 최종 검색 매칭 결과 리포팅
print(f"\n========================================================================")
print(f"  RAG 검색 매칭 요약 (실제 DB 조회 결과)")
print(f"  질의어: \"{test_query}\"")
print(f"========================================================================")
for idx, r in enumerate(search_results, 1):
    print(f"  [{idx}] {r['title']} (유사도 점수: {r['score']:.4f})")
    print(f"      arxiv ID  : {r['doc_id']}")
    print(f"      컨텐츠 샘플 : \"{str(r['text_chunk'])[:120]}...\"")
    print(f"------------------------------------------------------------------------")

🔎 RAG 검색 개시 | 도메인: cs (cs_embeddings) | 검색어: 'Neural network models and sequence alignment in computational biology'

  RAG 검색 매칭 요약 (실제 DB 조회 결과)
  질의어: "Neural network models and sequence alignment in computational biology"
  [1] Locality Sensitive Hashing-based Sequence Alignment Using Deep   Bidirectional LSTM Models (유사도 점수: 0.5706)
      arxiv ID  : 2004.02094
      컨텐츠 샘플 : "Title: Locality Sensitive Hashing-based Sequence Alignment Using Deep   Bidirectional LSTM Models

Abstract: Bidirection..."
------------------------------------------------------------------------
  [2] Optimizing scoring function of dynamic programming of pairwise profile   alignment using derivative free neural network (유사도 점수: 0.5564)
      arxiv ID  : 1708.09097
      컨텐츠 샘플 : "Title: Optimizing scoring function of dynamic programming of pairwise profile   alignment using derivative free neural n..."
------------------------------------------------------------------------


## 5단계. 실제 공통 파이프라인 모듈 호출해보기

실서비스 코드에 구현된 `api.common.rag_pipeline` 모듈을 직접 가져와(Import) 싱글톤 인스턴스로 조회를 테스트해봅니다.

In [5]:
# 앞단계에서 직접 작성한 로컬 함수(run_similarity_search) 호출 테스트
results = await run_similarity_search(domain="cs", query="genetic algorithm", k=1)
print(f"✅ 로컬 RAG 파이프라인 연동 성공! 검색된 논문 제목: {results[0]['title']}")

🔎 RAG 검색 개시 | 도메인: cs (cs_embeddings) | 검색어: 'genetic algorithm'
✅ 로컬 RAG 파이프라인 연동 성공! 검색된 논문 제목: Genetic optimization algorithms applied toward mission computability   models
